**Table of contents**<a id='toc0_'></a>    
- 1. [前言](#toc1_)    
- 2. [什么是FastAPI？](#toc2_)    
  - 2.1. [Pydantic 是什么？](#toc2_1_)    
  - 2.2. [Starlette 能干什么？](#toc2_2_)    
- 3. [在jupyter中运行fastapi](#toc3_)    
  - 3.1. [ 方法一：使用 nest_asyncio + uvicorn.run()（推荐）](#toc3_1_)    
  - 3.2. [方法二：在单独的线程中运行 FastAPI（适合多个服务并存）](#toc3_2_)    
- 4. [FastaAPI结构](#toc4_)    
  - 4.1. [路由](#toc4_1_)    
    - 4.1.1. [路由的基本用法](#toc4_1_1_)    
    - 4.1.2. [路由的类型](#toc4_1_2_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# 1. <a id='toc1_'></a>[前言](#toc0_)

# 2. <a id='toc2_'></a>[什么是FastAPI？](#toc0_)

```python
🌟 一句话解释：

FastAPI = Pydantic (数据验证) + Starlette (异步 Web 框架)
```

## 2.1. <a id='toc2_1_'></a>[Pydantic 是什么？](#toc0_)

- 📦 Pydantic 是 Python 中的一个强大库，用来做 数据验证 和 数据解析，它的核心理念是：✅ “让你的数据结构更安全、更可控，像写类一样自动校验数据类型和格式。”

- 🌟 Pydantic 能干嘛？
  1. 自动数据校验
你只需要用 Python 的类型注解写好模型，它会自动验证传进来的数据是否符合要求。

  2. 自动数据转换
比如你传入 "123"，它会自动变成整数 123；或把时间字符串变成 datetime 对象。

  3. 结构化数据管理
非常适合用来定义接口参数、配置文件、数据库模型、用户输入等数据结构。

-  🔥 Pydantic 的亮点

    |特性 | 说明|
    |---|---|
    |类型强校验 | 支持 str、int、float、bool、List、Dict、datetime 等等|
    |自动转换 | "1" → 1, "true" → True, "2023-01-01" → datetime|
    |嵌套模型 | 可以定义更复杂的嵌套数据结构|
    |易用性高 | 写起来就像写 Python 类，逻辑清晰|
    |高性能 | 用 Rust 写的底层（V2版），速度非常快|

In [1]:
from pydantic import BaseModel


class User(BaseModel):
    name: str
    age: int

user = User(name="Alice", age="25")  # 自动把 "25" 转成 int

print(user)  # User(name='Alice', age=25)

ModuleNotFoundError: No module named 'pydantic'

In [ ]:
# 如果你写错了：
User(name="Bob", age="abc")  # 会抛出 ValidationError 错误


## 2.2. <a id='toc2_2_'></a>[Starlette 能干什么？](#toc0_)

✅ 它本质上是一个ASGI 框架（异步 Web 框架），提供：

|功能模块 | 说明|
|---|---|
|📦 路由管理 | 支持 @route, @get, @post 等装饰器|
|⚡ 异步处理 | 支持 async def，无需额外配置|
|🧾 请求 & 响应对象 | 提供 Request / Response 类|
|🌐 中间件 | 支持添加权限验证、日志、跨域处理等|
|🗂️ 静态文件支持 | 可挂载 static/ 目录|
|🧪 测试工具 | 内建测试客户端，兼容 pytest|
|🌈 WebSocket | 原生支持异步双向通信（聊天室、推送等）|

🧠 它和 FastAPI 是什么关系？
FastAPI 实际上是基于 Starlette 做的二次封装：

|部分 | 来自哪里？|
|---|---|
|异步路由系统 | ✅ Starlette|
|Request, Response | ✅ Starlette|
|BackgroundTask / Middleware | ✅ Starlette|
|数据模型、参数验证 | ❌ FastAPI 自己集成 Pydantic|
|自动文档 / 类型提示 | ❌ FastAPI 专属亮点！|

🧠 所以说：FastAPI 是 Starlette 的「超进化版本」，专注于 API 结构化 + 类型安全。

📊 FastAPI vs Starlette 对比

功能 | Starlette | FastAPI|
|---|---|---|
|路由支持 | ✅ 有 | ✅ 有|
|异步支持 | ✅ 原生 | ✅ 基于 Starlette|
|数据验证 | ❌ 无 | ✅ 使用 Pydantic|
|自动文档 | ❌ 无 | ✅ OpenAPI & Swagger|
|使用场景 | 框架骨架、小应用 | 结构化 API、大型应用|

✅ 总结一句话：

Starlette 是 FastAPI 的异步 Web 框架底座，提供了高性能的路由、中间件、WebSocket、静态资源等功能，是 FastAPI 快速、强大、异步的基石。

# 3. <a id='toc3_'></a>[在jupyter中运行fastapi](#toc0_)

好问题！在 Jupyter Notebook 中启动 FastAPI 有点特别，因为 Jupyter 已经在运行一个事件循环了，而且它不是命令行终端，不能直接运行 uvicorn 命令。

## 3.1. <a id='toc3_1_'></a>[ 方法一：使用 nest_asyncio + uvicorn.run()（推荐）](#toc0_)

In [2]:
from fastapi import FastAPI 
import uvicorn
import nest_asyncio


app = FastAPI()


@app.get("/")
def read_root():
    return {"message": "Hello World"}


# 允许在已有事件循环中运行 uvicorn
nest_asyncio.apply()

# 启动服务器（不会卡住 Jupyter）
uvicorn.run(app, host="127.0.0.1", port=8000)

INFO:     Started server process [27032]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [27032]


In [4]:
def jupyter_run(app):
    '''封装函数，方便在 Jupyter Notebook 中运行 FastAPI'''
    import uvicorn
    import nest_asyncio
    # 允许在已有事件循环中运行 uvicorn
    nest_asyncio.apply()
    # 启动服务器（不会卡住 Jupyter）
    uvicorn.run(app, host="127.0.0.1", port=8000)

## 3.2. <a id='toc3_2_'></a>[方法二：在单独的线程中运行 FastAPI（适合多个服务并存）](#toc0_)

In [ ]:
from fastapi import FastAPI
import threading
import uvicorn


app = FastAPI()

@app.get("/")
def home():
    return {"msg": "Hello from thread!"}

def run():
    uvicorn.run(app, host="127.0.0.1", port=8000)

# 启动 FastAPI 服务线程
threading.Thread(target=run).start()


INFO:     Started server process [11788]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:8683 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:17686 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:17686 - "GET /openapi.json HTTP/1.1" 200 OK


# 4. <a id='toc4_'></a>[FastaAPI结构](#toc0_)

## 4.1. <a id='toc4_1_'></a>[路由](#toc0_)

FastAPI 的“路由”就是把“URL 地址”和你写的函数绑定起来，即：用户访问某个网址（URL），FastAPI 就调用你写好的函数来响应这个请求。

### 4.1.1. <a id='toc4_1_1_'></a>[路由的基本用法](#toc0_)

路由的基本写法：
- @app.get("/")：定义了一个“路由”规则

- 当用户访问 http://localhost:8000/，就会触发 homepage() 函数

In [9]:
from fastapi import FastAPI 


app = FastAPI()

@app.get("/") # 首页路由
def homepage():
    return {'message': 'Hello, World!'}

@app.get("/api/papers")
def get_papers():
    return {
    "2024": [
        {"title": "图神经网络", "authors": "张三等"},
    ]
}


jupyter_run(app)  # 在 Jupyter Notebook 中运行 FastAPI

INFO:     Started server process [27032]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [27032]


### 4.1.2. <a id='toc4_1_2_'></a>[路由的类型](#toc0_)

|装饰器 | 请求类型 | 用途|
|---|---|---|
|@app.get() | GET | 获取数据（页面、JSON 等）|
|@app.post() | POST | 提交数据（表单、上传）|
|@app.put() | PUT | 修改数据（更新）|
|@app.delete() | DELETE | 删除数据|
|@app.api_route() | 自定义支持多个方法 | 进阶用法|